In [1]:
!ls /kaggle/input

datasets


In [2]:
# Paths defined in next cell (single source)

In [3]:
TRAIN_DIR = "/kaggle/input/datasets/nehareddysabbidi/dataset/wYe7pBJ7-train/train"
TEST_DIR  = "/kaggle/input/datasets/nehareddysabbidi/dataset/Pa7a3Hin-test-public/Pa7a3Hin-test-public"

In [4]:
import os

print("TRAIN →", os.listdir(TRAIN_DIR)[:5])
print("TEST  →", os.listdir(TEST_DIR)[:5])

TRAIN → ['Scenario-A', 'Scenario-B']
TEST  → ['track_18210', 'track_15070', 'track_21876', 'track_21569', 'track_16337']


In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)


2.9.0+cpu
False
None


In [6]:
import cv2
import os

# take any sample image from train folder
sample_path = None

for root, _, files in os.walk(TRAIN_DIR):
    for f in files:
        if f.endswith(('.jpg', '.png')):
            sample_path = os.path.join(root, f)
            break
    if sample_path:
        break

img = cv2.imread(sample_path)
h, w, _ = img.shape

print("Sample image:", sample_path)
print("Width :", w)
print("Height:", h)


Sample image: /kaggle/input/datasets/nehareddysabbidi/dataset/wYe7pBJ7-train/train/Scenario-A/Mercosur/track_08602/lr-001.png
Width : 45
Height: 20


In [7]:
import cv2, os

TEST_DIR = "/kaggle/input/datasets/nehareddysabbidi/dataset/Pa7a3Hin-test-public/Pa7a3Hin-test-public"

first5 = ["track_10005", "track_10015", "track_10016", "track_10035", "track_10039"]

for track in first5:
    path = os.path.join(TEST_DIR, track)
    
    print("\n==============================")
    print("TRACK:", track)
    print("==============================")
    
    for img in sorted(os.listdir(path)):
        img_path = os.path.join(path, img)
        im = cv2.imread(img_path)
        h, w, _ = im.shape
        
        tag = "LR" if "LR" in img or "lr" in img else "HR"
        print(f"{img:25s} → {tag} → {w} × {h}")



TRACK: track_10005
lr-001.jpg                → LR → 44 × 15
lr-002.jpg                → LR → 44 × 15
lr-003.jpg                → LR → 44 × 16
lr-004.jpg                → LR → 44 × 16
lr-005.jpg                → LR → 45 × 16

TRACK: track_10015
lr-001.jpg                → LR → 45 × 18
lr-002.jpg                → LR → 48 × 19
lr-003.jpg                → LR → 48 × 19
lr-004.jpg                → LR → 49 × 19
lr-005.jpg                → LR → 49 × 19

TRACK: track_10016
lr-001.jpg                → LR → 50 × 18
lr-002.jpg                → LR → 50 × 18
lr-003.jpg                → LR → 51 × 18
lr-004.jpg                → LR → 52 × 18
lr-005.jpg                → LR → 53 × 19

TRACK: track_10035
lr-001.jpg                → LR → 48 × 18
lr-002.jpg                → LR → 48 × 18
lr-003.jpg                → LR → 49 × 18
lr-004.jpg                → LR → 49 × 18
lr-005.jpg                → LR → 49 × 18

TRACK: track_10039
lr-001.jpg                → LR → 46 × 17
lr-002.jpg                → LR → 45 × 1

In [8]:
# Redundant: track structure covered in Cell 11 (BASE was undefined here)


In [9]:
!pip install torch torchvision opencv-python pillow tqdm timm

In [10]:
!git clone https://github.com/JingyunLiang/SwinIR.git

Cloning into 'SwinIR'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 333 (delta 6), reused 2 (delta 2), pack-reused 323 (from 2)
Receiving objects: 100% (333/333), 29.84 MiB | 40.15 MiB/s, done.
Resolving deltas: 100% (119/119), done.


In [11]:
import sys
sys.path.append("/kaggle/working/SwinIR")

from models.network_swinir import SwinIR
import torch


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [12]:
import os, json, random
from sklearn.model_selection import train_test_split

BASE = TRAIN_DIR

tracks = []
for scenario in ["Scenario-A","Scenario-B"]:
    for country in ["Brazilian","Mercosur"]:
        p = os.path.join(BASE,scenario,country)
        tracks += [os.path.join(p,t) for t in os.listdir(p)]

train_tracks, val_tracks = train_test_split(tracks, test_size=0.1, random_state=42)

print("Train tracks:", len(train_tracks))
print("Val tracks:", len(val_tracks))


Train tracks: 18000
Val tracks: 2000


In [13]:
import cv2, torch
from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self, track_list):
        self.data = []

        for tp in track_list:
            ann = json.load(open(os.path.join(tp,"annotations.json")))
            text = ann["plate"]

            for i in range(1,6):
                img = os.path.join(tp,f"hr-00{i}.png")
                self.data.append((img,text))

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        img_p,label = self.data[idx]

        img = cv2.imread(img_p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # normalize for PARSeq
        img = cv2.resize(img,(128,32))
        img = torch.tensor(img).permute(2,0,1)/255.0

        return img,label


In [14]:
!pip install torch torchvision
!pip install git+https://github.com/baudm/parseq.git

  Cloning https://github.com/baudm/parseq.git to /tmp/pip-req-build-1khmvd0e
  Running command git clone --filter=blob:none --quiet https://github.com/baudm/parseq.git /tmp/pip-req-build-1khmvd0e
  Resolved https://github.com/baudm/parseq.git to commit 1902db043c029a7e03a3818c616c06600af574be
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for strhub: filename=strhub-1.2.0-py3-none-any.whl size=65416 sha256=3196c16beebfdca56f4e7789807e5c0bc669687ae76da00d9a9a785b8655579c
  Stored in directory: /tmp/pip-ephem-wheel-cache-kjeqt1cv/wheels/e9/35/35/eef8664876b6bd9efc71cca6dd8592e606a58e98a30dbfa2d4
Successfully built strhub


In [15]:
import os
import cv2
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm

In [16]:
class CRNN(nn.Module):
    def __init__(self, num_classes=36):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),        # 32→16

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),        # 16→8

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2,1)),        # 8→4

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d((2,1))         # 4→2
        )

        # ✅ CORRECT INPUT = 512*2 = 1024
        self.rnn = nn.Sequential(
            nn.LSTM(1024, 256, bidirectional=True, batch_first=True),
            nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        )

        self.fc = nn.Linear(512, num_classes + 1)


    def forward(self, x):
        f = self.cnn(x)

        b, c, h, w = f.size()      # h should be 2

        f = f.permute(0, 3, 1, 2)  # [B,W,C,H]
        f = f.view(b, w, c*h)      # → 1024

        o,_ = self.rnn[0](f)
        o,_ = self.rnn[1](o)

        return self.fc(o)

In [17]:
CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

ctoi = {c:i+1 for i,c in enumerate(CHARS)}   # 0 = blank
itoc = {i+1:c for i,c in enumerate(CHARS)}

def encode_text(text):

    text = text.upper()

    # remove spaces and unexpected symbols
    text = text.replace(" ", "").replace("-", "")

    # keep only allowed characters
    clean = [c for c in text if c in ctoi]

    return [ctoi[c] for c in clean]


def decode_pred(seq):
    s = ""
    prev = 0
    for i in seq:
        if i != prev and i != 0:
            s += itoc.get(i,"")
        prev = i

    return s


In [18]:
class PlateDataset(Dataset):
    def __init__(self, tracks):
        self.samples = []

        for tp in tracks:
            ann = json.load(open(os.path.join(tp, "annotations.json")))
            text = ann["plate_text"]

            for i in range(1,6):

                png = os.path.join(tp, f"hr-00{i}.png")
                jpg = os.path.join(tp, f"hr-00{i}.jpg")

                img = png if os.path.exists(png) else jpg

                self.samples.append((img, text))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):

        img_p, label = self.samples[idx]

        img = cv2.imread(img_p)

        if img is None:
            raise ValueError(f"Image not found → {img_p}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (128,32))

        img = torch.tensor(img).permute(2,0,1)/255.0

        return img, label


In [19]:
train_loader = DataLoader(
    PlateDataset(train_t),
    batch_size=32,
    shuffle=True
)

NameError: name 'train_t' is not defined

In [ ]:
ctc = nn.CTCLoss(blank=0, zero_infinity=True)

def prepare_batch(labels, device):

    targets=[]
    lengths=[]

    for t in labels:
        e=encode_text(t)
        targets+=e
        lengths.append(len(e))

    return torch.tensor(targets).to(device), \
           torch.tensor(lengths)


In [ ]:
def eval_val():

    model.eval()

    preds, gts = [], []

    for tp in val_t:

        ann = json.load(open(tp+"/annotations.json"))
        gt = ann["plate_text"]

        track_preds = []

        for i in range(1,6):

            # ---- load LR directly ----
            png = f"{tp}/lr-00{i}.png"
            jpg = f"{tp}/lr-00{i}.jpg"

            lr = png if os.path.exists(png) else jpg

            img = cv2.imread(lr)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            img = cv2.resize(img,(128,32))

            t = torch.tensor(img).permute(2,0,1)/255.0
            t = t.unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(t)

            seq = out.argmax(2)[0].cpu().numpy()
            text = decode_pred(seq)

            track_preds.append(text)

        # majority vote over 5 frames
        final = Counter(track_preds).most_common(1)[0][0]

        preds.append(final)
        gts.append(gt)

    acc = sum(p==g for p,g in zip(preds,gts)) / len(gts)

    print("VAL Plate Accuracy (RAW LR):", round(acc*100,2), "%")


In [ ]:
model = CRNN().to(device)
optimizer = torch.optim.Adam(model.parameters(),1e-4)

In [ ]:
device="cuda"

model=CRNN().to(device)
optimizer=torch.optim.Adam(model.parameters(),1e-4)

EPOCHS=15

for epoch in range(EPOCHS):

    model.train()

    loop=tqdm(train_loader)

    for imgs,labels in loop:

        imgs=imgs.to(device)

        out=model(imgs)

        log_probs=out.log_softmax(2)

        b,w,c=log_probs.shape

        input_lengths=torch.full((b,),w)

        targets,target_lengths=prepare_batch(labels,device)

        loss=ctc(log_probs.permute(1,0,2),
                 targets,
                 input_lengths,
                 target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

    torch.save(model.state_dict(),f"crnn_{epoch}.pth")

    eval_val()


In [ ]:
torch.save(model.state_dict(), "/kaggle/working/FINAL_epoch15.pth")
print("Final model saved!")

import os
print(os.listdir("/kaggle/working"))

In [ ]:
!zip -r /kaggle/working/ocr_models.zip /kaggle/working/*.pth

In [ ]:
!cp /kaggle/working/ocr_models.zip /kaggle/working/download_ocr_models.zip

In [ ]:
# Working dir listing in Cell 32

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/ocr_models.zip')

In [ ]:
import torch, os, json, cv2
from collections import Counter
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate_model(model_path):

    model = CRNN().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    preds, gts = [], []

    for tp in tqdm(val_t, desc=model_path.split("/")[-1]):

        ann = json.load(open(tp+"/annotations.json"))
        gt = ann["plate_text"]

        track_preds = []

        for i in range(1,6):

            png=f"{tp}/hr-00{i}.png"
            jpg=f"{tp}/hr-00{i}.jpg"

            imgp = png if os.path.exists(png) else jpg

            img=cv2.imread(imgp)
            img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
            img=cv2.resize(img,(128,32))

            t=torch.tensor(img).permute(2,0,1)/255.0
            t=t.unsqueeze(0).to(device)

            with torch.no_grad():
                out=model(t)

            seq=out.argmax(2)[0].cpu().numpy()
            text=decode_pred(seq)

            track_preds.append(text)

        final=Counter(track_preds).most_common(1)[0][0]

        preds.append(final)
        gts.append(gt)

    acc=sum(p==g for p,g in zip(preds,gts))/len(gts)
    return acc


In [ ]:
results = {}

for f in sorted(os.listdir("/kaggle/working")):
    if f.startswith("crnn_") and f.endswith(".pth"):
        path = "/kaggle/working/" + f
        acc = evaluate_model(path)
        results[f] = acc
        print(f, "→", round(acc*100,2), "%")


In [ ]:
best = max(results, key=results.get)

print("\n===== BEST CHECKPOINT =====")
print("Model :", best)
print("Accuracy :", round(results[best]*100,2), "%")


In [ ]:
!mv /kaggle/working/crnn_12.pth /kaggle/working/BEST_OCR.pth
print("Renamed successfully!")


In [ ]:
!ls -lh /kaggle/working

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/BEST_OCR.pth')

In [ ]:
import os

keep = "BEST_OCR.pth"

for f in os.listdir("/kaggle/working"):
    if f.endswith(".pth") and f != keep:
        os.remove("/kaggle/working/" + f)
        print("Deleted →", f)

print("\nRemaining files:")
!ls -lh /kaggle/working

In [ ]:
!rm -f /kaggle/working/*.zip

In [ ]:
# BEST_OCR.pth in ocr-model

In [ ]:
# Imports from Cell 14


In [ ]:
# CRNN defined in Cell 15


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN().to(device)

model.load_state_dict(
    torch.load("/kaggle/input/ocr-model/BEST_OCR.pth",
               map_location=device)
)

model.eval()

print("✅ Your BEST OCR restored!")


In [ ]:
# enhance() defined in Cell 64 (requires sr_model from Cell 63)


In [ ]:
# Input listing in Cell 0

In [ ]:
import os

BASE = "/kaggle/input/datasets/nehareddysabbidi/dataset/wYe7pBJ7-train/train"

train_t = []

for scenario in ["Scenario-A", "Scenario-B"]:
    for country in ["Brazilian", "Mercosur"]:

        path = os.path.join(BASE, scenario, country)

        for track in os.listdir(path):
            train_t.append(os.path.join(path, track))


print("Total TRAIN tracks:", len(train_t))


In [ ]:
CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# original mapping 0–35
ctoi = {c:i for i,c in enumerate(CHARS)}
itoc = {i:c for i,c in enumerate(CHARS)}

# ---- PATCH: accept unwanted symbols as BLANK (index 36) ----
BLANK = 36
ctoi[" "] = BLANK       # space
ctoi["-"] = BLANK       # hyphen
ctoi["."] = BLANK       # just in case

def encode_text(text):
    text = text.upper()
    return [ctoi[c] for c in text if c in ctoi]


def decode_pred(seq):
    s = ""
    prev = -1
    for i in seq:
        # ignore blank=36
        if i != prev and i < len(CHARS):
            s += itoc.get(i, "")
        prev = i
    return s


In [ ]:
from tqdm import tqdm
import json, os, cv2, torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- OCR on single LR image ----------
def ocr_lr(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128,32))

    t = torch.tensor(img).permute(2,0,1).float()/255.
    t = t.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(t)
        prob = out.softmax(2)

        pred = prob.argmax(2)[0].cpu().numpy()
        conf = prob.max(2)[0].mean().item()

    return decode_pred(pred), conf


# ---------- Track level fusion ----------
def predict_track(tp):

    ann = json.load(open(tp+"/annotations.json"))
    gt = ann["plate_text"]

    preds = []
    confs = []

    for i in range(1,6):

        p1 = f"{tp}/lr-00{i}.png"
        p2 = f"{tp}/lr-00{i}.jpg"
        p  = p1 if os.path.exists(p1) else p2

        txt, c = ocr_lr(p)

        preds.append(txt)
        confs.append(c)

    # majority + confidence
    final = max(
        set(preds),
        key=lambda x: preds.count(x) + sum(c for t,c in zip(preds,confs) if t==x)
    )

    return final, gt


# ---------- FULL TRAIN EVALUATION ----------
def full_train_accuracy(tracks):

    total = len(tracks)
    correct = 0

    for tp in tqdm(tracks):

        pred, gt = predict_track(tp)

        if pred == gt:
            correct += 1

    acc = 100 * correct / total

    print("\n==============================")
    print(f"TOTAL TRACKS : {total}")
    print(f"CORRECT      : {correct}")
    print(f"LR ACCURACY  : {acc:.2f}%")
    print("==============================")

    return acc


# ===== RUN ON ENTIRE TRAIN SET =====
full_train_accuracy(train_t)


In [ ]:
import random
random.shuffle(train_t)

split = int(0.9 * len(train_t))

ft_train = train_t[:split]
ft_val   = train_t[split:]

print("Train:", len(ft_train))
print("Val  :", len(ft_val))


In [ ]:
class LRFineTuneDataset(torch.utils.data.Dataset):
    def __init__(self, tracks):
        self.samples = []

        for tp in tracks:
            ann = json.load(open(tp+"/annotations.json"))
            text = ann["plate_text"]

            for i in range(1,6):
                p1 = f"{tp}/lr-00{i}.png"
                p2 = f"{tp}/lr-00{i}.jpg"
                p  = p1 if os.path.exists(p1) else p2

                self.samples.append((p, text))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        path, text = self.samples[idx]

        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img,(128,32))

        # --- augment ---
        if torch.rand(1) < 0.5:
            img = cv2.GaussianBlur(img,(3,3),0)

        if torch.rand(1) < 0.3:
            img = img + np.random.normal(0,8,img.shape)

        img = np.clip(img,0,255)

        t = torch.tensor(img).permute(2,0,1).float()/255.

        return t, text


In [ ]:
def ocr_lr_tensor(t):
    with torch.no_grad():
        out = model(t.unsqueeze(0))
        pred = out.softmax(2).argmax(2)[0].cpu().numpy()
    return decode_pred(pred)


def val_accuracy():

    model.eval()

    correct=0
    total=0

    for tp in tqdm(ft_val):

        ann=json.load(open(tp+"/annotations.json"))
        gt=ann["plate_text"]

        preds=[]

        for i in range(1,6):
            p1=f"{tp}/lr-00{i}.png"
            p2=f"{tp}/lr-00{i}.jpg"
            p=p1 if os.path.exists(p1) else p2

            img=cv2.imread(p)
            img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
            img=cv2.resize(img,(128,32))

            t=torch.tensor(img).permute(2,0,1).float()/255.
            t=t.to(device)

            preds.append(ocr_lr_tensor(t))

        final=max(set(preds),key=preds.count)

        if final==gt:
            correct+=1
        total+=1

    acc=100*correct/total
    print("VAL LR ACC:",acc,"%")
    return acc


In [ ]:
train_dataset = LRFineTuneDataset(ft_train)
val_dataset   = LRFineTuneDataset(ft_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), 1e-5)
ctc = nn.CTCLoss(blank=36, zero_infinity=True)

best = 0

for epoch in range(6):

    model.train()

    for imgs, labels in tqdm(train_loader):

        imgs = imgs.to(device)

        out = model(imgs)
        log_probs = out.log_softmax(2)

        b,w,c = log_probs.shape
        input_lengths = torch.full((b,), w)

        targets=[]
        lengths=[]

        for t in labels:
            e=[ctoi[c] for c in t]
            targets+=e
            lengths.append(len(e))

        targets=torch.tensor(targets).to(device)
        target_lengths=torch.tensor(lengths)

        loss = ctc(log_probs.permute(1,0,2),
                   targets,
                   input_lengths,
                   target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    print("\nEpoch",epoch,"VALIDATION")
    acc = val_accuracy()

    torch.save(model.state_dict(),f"LR_FINETUNE_{epoch}.pth")

    if acc>best:
        best=acc
        torch.save(model.state_dict(),"BEST_LR_OCR.pth")
        print("NEW BEST SAVED 🚀")


In [ ]:
device='cpu'
model = CRNN().to(device)

model.load_state_dict(
    torch.load("/kaggle/input/best-lr-ocr/BEST_LR_OCR.pth", map_location=device)
)

print("Loaded BEST_LR_OCR.pth successfully")


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), 1e-5)
ctc = nn.CTCLoss(blank=36, zero_infinity=True)

best = 37.7   # start from previous best

for epoch in range(6, 15):   # continue from epoch 6

    model.train()

    for imgs, labels in tqdm(train_loader):

        imgs = imgs.to(device)

        out = model(imgs)
        log_probs = out.log_softmax(2)

        b,w,c = log_probs.shape
        input_lengths = torch.full((b,), w)

        targets=[]
        lengths=[]

        for t in labels:
            e=[ctoi[c] for c in t if c in ctoi]
            targets+=e
            lengths.append(len(e))

        targets=torch.tensor(targets).to(device)
        target_lengths=torch.tensor(lengths)

        loss = ctc(log_probs.permute(1,0,2),
                   targets,
                   input_lengths,
                   target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("\nEpoch",epoch,"VALIDATION")
    acc = val_accuracy()

    torch.save(model.state_dict(),f"LR_FINETUNE_{epoch}.pth")

    if acc>best:
        best=acc
        torch.save(model.state_dict(),"BEST_LR_OCR.pth")
        print("NEW BEST SAVED 🚀")


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN().to(device)
model.load_state_dict(torch.load("BEST_LR_OCR.pth", map_location=device))
model.eval()

print("Best LR model loaded ✅")


In [ ]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm

def ocr_lr(path):

    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(128,32))
    img = img.astype("float32")/255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(img)
        prob = out.softmax(2)
        pred = prob.argmax(2)[0].cpu().numpy()

    return decode_pred(pred)


def predict_track(tp):

    ann = json.load(open(tp+"/annotations.json"))
    gt = ann["plate_text"]

    preds = []

    for i in range(1,6):

        p1 = f"{tp}/lr-00{i}.png"
        p2 = f"{tp}/lr-00{i}.jpg"

        p = p1 if os.path.exists(p1) else p2

        txt = ocr_lr(p)
        preds.append(txt)

    # Majority vote
    final_pred = max(set(preds), key=preds.count)

    return final_pred, gt


In [ ]:
all_tracks = train_t  # from Cell 43


In [ ]:
def full_train_accuracy(tracks):

    correct = 0

    for tp in tqdm(tracks):

        pred, gt = predict_track(tp)

        if pred == gt:
            correct += 1

    acc = correct / len(tracks) * 100

    print("\n==============================")
    print("TOTAL TRACKS :", len(tracks))
    print("CORRECT      :", correct)
    print("LR ACCURACY  :", round(acc,2), "%")
    print("==============================")

    return acc


full_train_accuracy(all_tracks)


In [ ]:
# best-lr-ocr contains BEST_LR_OCR.pth

In [ ]:
# Imports from Cell 14

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN().to(device)

model.load_state_dict(
    torch.load("/kaggle/input/best-lr-ocr/BEST_LR_OCR.pth",
               map_location=device)
)

model.eval()

print("Model Loaded Successfully")


In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from collections import Counter

TEST_BASE = "/kaggle/input/datasets/nehareddysabbidi/test-set/TKzFBtn7-test-blind"

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

def ocr_lr(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(128,32))
    img = torch.tensor(img).permute(2,0,1).float()/255
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(img)
        prob = out.softmax(2)
        pred = prob.argmax(2)[0].cpu().numpy()
        conf = prob.max(2)[0].mean().item()

    return decode_pred(pred), conf


submission_lines = []

tracks = sorted(os.listdir(TEST_BASE))

for track in tqdm(tracks):

    track_path = os.path.join(TEST_BASE, track)

    preds = []
    confs = []

    for i in range(1,6):

        p1 = f"{track_path}/lr-00{i}.png"
        p2 = f"{track_path}/lr-00{i}.jpg"

        path = p1 if os.path.exists(p1) else p2

        txt, conf = ocr_lr(path)

        preds.append(txt)
        confs.append(conf)

    # Majority voting
    final_text = Counter(preds).most_common(1)[0][0]

    # confidence = average of images that predicted final_text
    selected_confs = [c for p,c in zip(preds,confs) if p == final_text]
    final_conf = np.mean(selected_confs) if selected_confs else 0.0

    submission_lines.append(f"{track},{final_text};{final_conf:.4f}")


# Save submission file
with open("submission.txt","w") as f:
    for line in submission_lines:
        f.write(line+"\n")

print("Submission file created ✅")


In [ ]:
!head -20 submission.txt

In [ ]:
!zip submission.zip submission.txt

In [ ]:
import torch
from SwinIR.models.network_swinir import SwinIR as net

device = "cuda" if torch.cuda.is_available() else "cpu"

# create model
sr_model = net(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6,6,6,6,6,6],
    embed_dim=180,
    num_heads=[6,6,6,6,6,6],
    mlp_ratio=2,
    upsampler='nearest+conv',
    resi_connection='1conv'
).to(device)

# load weights
weights = torch.load(
    "/kaggle/input/swinir-weights/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth",
    map_location=device
)

sr_model.load_state_dict(weights, strict=False)
sr_model.eval()

print("✅ SwinIR Loaded Successfully!")


In [ ]:
import cv2, numpy as np, json, torch
from collections import Counter
from matplotlib import pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# ===== IMAGE LOADER =====
def load_img(p):
    img = cv2.imread(p)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


# ===== SWINIR ENHANCE =====
def enhance(path):
    img = load_img(path)
    img = img.astype(np.float32)/255.0
    t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        sr = sr_model(t)

    sr = sr.squeeze(0).permute(1,2,0).cpu().numpy()
    sr = np.clip(sr*255,0,255).astype(np.uint8)
    return sr


# ===== OCR PREDICT =====
def ocr_pred(img):
    img = cv2.resize(img,(128,32))
    img = img.astype(np.float32)/255.0
    t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        o = model(t)
        o = o.log_softmax(2)
        o = o.permute(1,0,2)

    seq = o.argmax(2)[:,0].cpu().numpy()
    return decode_pred(seq)


# ===== SHOW PER TRACK =====
def show_track(tp):

    ann = json.load(open(tp+"/annotations.json"))
    gt = ann["plate_text"]

    print("\n================================")
    print("TRACK:", tp.split("/")[-1])
    print("GROUND TRUTH:", gt)

    hr_preds=[]
    sr_preds=[]

    for i in range(1,6):

        # PATHS
        hpng=f"{tp}/hr-00{i}.png"
        hjpg=f"{tp}/hr-00{i}.jpg"
        lpng=f"{tp}/lr-00{i}.png"
        ljpg=f"{tp}/lr-00{i}.jpg"

        hrp = hpng if os.path.exists(hpng) else hjpg
        lrp = lpng if os.path.exists(lpng) else ljpg

        hr = load_img(hrp)
        sr = enhance(lrp)

        ph = ocr_pred(hr)
        ps = ocr_pred(sr)

        hr_preds.append(ph)
        sr_preds.append(ps)

        print(f"\nImage {i}")
        print("HR OCR:", ph)
        print("SR OCR:", ps)

    # voting
    hr_final = Counter(hr_preds).most_common(1)[0][0]
    sr_final = Counter(sr_preds).most_common(1)[0][0]

    print("\n---- FINAL VOTING ----")
    print("HR FINAL:", hr_final)
    print("SR FINAL:", sr_final)
    print("GT     :", gt)

    print("HR CORRECT:", hr_final==gt)
    print("SR CORRECT:", sr_final==gt)


In [ ]:
tracks = val_t[:5]   # first 5 validation tracks

for tp in tracks:
    show_track(tp)